# HippoVoice — Colab Runner (T4 GPU)

**v1 — updated 2026-07-02**

**Runtime:** Runtime → Change runtime type → **T4 GPU**

## Startup sequence (fresh runtime)
1. Run **Step 1** (install deps) → takes ~5 min
2. **Runtime → Restart session** (Ctrl+M .)
3. Run **Step 2** (clone + autoreload) → run this again after any push
4. Run sections in order below

| Section | Models loaded | VRAM |
|---------|--------------|------|
| CPU tests | none | 0 GB |
| LLM smoke test | Qwen3-4B 4-bit | ~3 GB |
| Signal/noise benchmark | Qwen3-4B 4-bit | ~3 GB |
| LoCoMo benchmark | Qwen3-4B 4-bit | ~3 GB |
| Voice test | Canary + Qwen3-4B 4-bit → swap Fish TTS | ~8 GB |

In [ ]:
# ── STEP 1: Install deps ─────────────────────────────────────────────────────
# Lightweight stack — runs on CPU, T4, Kaggle, Mac.
# No NeMo, no Fish Speech, no Rust. ~2 min install.
# Run once, then Runtime → Restart session.

!pip install -q openai-whisper          # STT (~150MB model, works everywhere)
!pip install -q pyttsx3                 # TTS (offline, no model download)
!pip install -q chromadb networkx "sentence-transformers>=2.7.0"
!pip install -q soundfile jiwer datasets pytest pytest-mock
!pip install -q bitsandbytes accelerate  # 4-bit quant on CUDA (ignored on CPU/Mac)

# Pin last — other packages silently downgrade these
!pip install -q --force-reinstall "numpy==2.0.2" "protobuf==5.29.6"

import numpy as np, google.protobuf
print(f'numpy     {np.__version__}')
print(f'protobuf  {google.protobuf.__version__}')
print()
print('━' * 50)
print('NOW: Runtime → Restart session  (Ctrl+M .)')
print('THEN: run Step 2 and below.')
print('━' * 50)

In [ ]:
# ── STEP 2: Clone repo ────────────────────────────────────────────────────────
# Run after restart. Re-run this cell after any git push.
# Private repo: add your GitHub PAT to Kaggle Secrets as GH_TOKEN, then enable it.

import os, sys

if os.path.exists('/kaggle/working'):
    REPO_DIR = '/kaggle/working/hippovoice'
    # Load PAT from Kaggle Secrets if available
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GH_TOKEN')
        CLONE_URL = f'https://{token}@github.com/shivansh193/hippovoice.git'
    except Exception:
        CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'
        print('No GH_TOKEN secret found — will work for public repos only')
else:
    REPO_DIR = '/content/hippovoice'
    CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull
else:
    !git clone {CLONE_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    %load_ext autoreload
    %autoreload 2
    print('autoreload active')
except Exception:
    print('autoreload unavailable')

import numpy as np, subprocess
try:
    commit = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h'], text=True).strip()
except Exception:
    commit = 'unknown'
print(f'numpy {np.__version__}  |  commit [{commit}]')
print('Ready.')

## DRY RUN MODE — mocked models (no GPU, no downloads)

Run the 3 cells below **before** any section past this point to replace the
LLM, STT, and TTS loaders with hardcoded stubs. This validates the full
pipeline (memory extraction → store → retrieve → generate → synthesize)
end-to-end with zero model downloads and zero GPU.

Each mock monkeypatches the loader function/class in place, so every
downstream cell (`LLMClient()`, `load_whisper()`, `load_tts()`, etc.) is
**unchanged** — it just receives a mock object instead of a real model.

**Skip these 3 cells entirely to run the notebook for real.**

In [ ]:
# ── DRY RUN 1/3: mock LLM ──────────────────────────────────────────────────
import json
import llm.client as _llm_client_mod

class MockLLMClient:
    """Hardcoded-response stand-in — same public API as llm.client.LLMClient."""
    model_name = "mock-llm"
    _backend = "mock"

    def __init__(self, *a, **kw):
        pass

    def generate(self, system: str, messages: list[dict], max_tokens: int = 512) -> str:
        sys_l = (system or "").lower()
        user_content = messages[-1]["content"] if messages else ""
        if "memory extraction" in sys_l:
            # memory.extractor.EXTRACTION_PROMPT always ends with "Turn: {turn}"
            turn_text = user_content.split("Turn: ", 1)[-1].strip()
            return json.dumps([{"content": turn_text, "entity": "user", "type": "event"}])
        if "summarise these facts" in sys_l:
            return "Mock summary of several low-salience facts."
        if "2 + 2" in user_content:
            return "The answer is 4."
        return "Mock response acknowledging: " + user_content[:60]

    def unload(self):
        pass

_llm_client_mod.LLMClient = MockLLMClient
print('LLM mocked — LLMClient() now returns MockLLMClient')

In [ ]:
# ── DRY RUN 2/3: mock STT (whisper) ────────────────────────────────────────
import numpy as np
import stt.model as _stt_model_mod
import stt.transcribe as _stt_transcribe_mod

def _mock_load_whisper(model_size: str = "tiny"):
    return {"_mock": True, "size": model_size}

def _mock_transcribe_with_embedding(model, audio_path: str):
    # whisper-tiny's encoder is 384-dim; only shape/dtype matter downstream
    # (tag_emotion_audio only reads np.std()), so the audio file is never opened.
    emb = np.random.RandomState(0).randn(384).astype(np.float32)
    return "This is a mocked transcript for dry-run validation.", emb

_stt_model_mod.load_whisper = _mock_load_whisper
_stt_model_mod.load_canary = _mock_load_whisper
_stt_transcribe_mod.transcribe_with_embedding = _mock_transcribe_with_embedding

# Placeholder so the STT+generation cell doesn't NameError if the mic-recording
# cell is skipped — the mock above never reads this file.
AUDIO_PATH = '/content/user_input.wav'

print('STT mocked — load_whisper()/load_canary() return a mock model, transcribe_with_embedding() returns a fixed transcript')

In [ ]:
# ── DRY RUN 3/3: mock TTS (pyttsx3) ────────────────────────────────────────
import numpy as np
import soundfile as sf
import tts.model as _tts_model_mod

class MockTTSEngine:
    """Stand-in pyttsx3 engine — implements the two methods tts.synthesize.synthesize() calls."""
    def save_to_file(self, text, path):
        sr = 22050
        silence = np.zeros(int(sr * 0.3), dtype=np.float32)
        sf.write(path, silence, sr)

    def runAndWait(self):
        pass

def _mock_load_tts(rate: int = 175, volume: float = 1.0):
    return MockTTSEngine()

_tts_model_mod.load_tts = _mock_load_tts
_tts_model_mod.load_fish_tts = _mock_load_tts

print('TTS mocked — load_tts()/load_fish_tts() return a mock engine that writes a short silent WAV')

In [ ]:
# Verify salience math
from memory.scorer import compute_salience

checks = [
    ('fear fresh',      compute_salience(1.0, {'label': 'fear',    'intensity': 0.95}, 0,  0), '> 3.0'),
    ('neutral 45 turns',compute_salience(1.0, {'label': 'neutral', 'intensity': 0.01}, 0, 45), '< 0.25'),
    ('fear 45 turns',   compute_salience(1.0, {'label': 'fear',    'intensity': 0.90}, 0, 45), '> 0.25'),
]
for label, val, expectation in checks:
    print(f'{label:20s}: {val:.4f}  ({expectation})')

## 2. Memory Core + Store Tests

These need sentence-transformers (CPU, no GPU).

In [ ]:
!pytest tests/test_store.py tests/test_retriever.py -v --tb=short

In [ ]:
# Manual memory walkthrough to sanity-check the full stack without LLM
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve
import numpy as np

mem = HippoMemory()
entries = [
    ('My dog Max got diagnosed with cancer. I am devastated.',  'sadness', 0.9),
    ('The weather was cloudy today.',                           'neutral', 0.05),
    ('Max had his first chemo session. I held him the whole time.', 'sadness', 0.85),
    ('I had cereal for breakfast.',                             'neutral', 0.05),
]

for i, (content, label, intensity) in enumerate(entries):
    mem.add({'content': content,
             'emotion': {'label': label, 'intensity': intensity},
             'base_weight': 1.0, 'recall_count': 0, 'turn_created': i})

print(f'Stored {mem.count()} memories\n')
print('Query: "tell me about Max"')
results = hippo_retrieve('tell me about Max', mem, mem.graph, current_turn=4, top_k=4)
for r in results:
    print(f'  [{r["current_salience"]:.3f}] {r["content"]}')

## 3. Load LLM (Qwen3-4B, 4-bit — fits in ~3GB VRAM)

This is the only model needed for benchmarks.

In [ ]:
import torch
device_name = 'T4' if torch.cuda.is_available() else 'CPU'
if torch.cuda.is_available():
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f'VRAM free before: {free:.1f} GB  ({device_name})')
else:
    print(f'Running on CPU')

from llm.client import LLMClient
# Default: Qwen3-0.6B (~600MB) — runs on CPU or T4 with no quantization needed.
# To upgrade: LLMClient(model_name='Qwen/Qwen3-4B', load_in_4bit=True)
llm = LLMClient()
print(f'Loaded: {llm.model_name}  backend: {llm._backend}')

In [ ]:
# Smoke test
resp = llm.generate(
    system='Answer in one sentence.',
    messages=[{'role': 'user', 'content': 'What is 2 + 2?'}],
    max_tokens=30
)
print('Response:', resp)
assert '4' in resp
print('LLM working')

## 4. Signal/Noise Benchmark (Core Research Claim)

In [ ]:
from pipeline import HippoVoicePipeline
from baselines.naive_rag import NaiveRAG
from benchmarks.signal_noise.run import run_signal_noise_benchmark

print('Running signal/noise benchmark (~5 min)...\n')

hippo = HippoVoicePipeline(llm_client=llm, text_only=True)
naive = NaiveRAG()

hippo_result = run_signal_noise_benchmark(hippo, 'HippoVoice')
naive_result  = run_signal_noise_benchmark(naive,  'NaiveRAG')

print('=' * 50)
print(f'HippoVoice  noise rate: {hippo_result["noise_rate"]:.1%}  (signal={hippo_result["signal_count"]}, noise={hippo_result["noise_count"]})')
print(f'Naive RAG   noise rate: {naive_result["noise_rate"]:.1%}  (signal={naive_result["signal_count"]}, noise={naive_result["noise_count"]})')
print('=' * 50)
print(f'HippoVoice < 10% noise: {"PASS" if hippo_result["noise_rate"] < 0.10 else "FAIL"}')
print(f'Naive RAG  > 40% noise: {"PASS" if naive_result["noise_rate"]  > 0.40 else "FAIL"}')

## 5. LoCoMo Benchmark

~20 min on T4 with Qwen3-4B 4-bit.

In [ ]:
import torch, gc

# Unload LLM if still in memory
if 'llm' in dir() and llm is not None:
    llm.unload()
    llm = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f'VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')
else:
    print('Running on CPU')

In [ ]:
from stt.model import load_whisper
stt_model = load_whisper("tiny")   # ~150MB, upgrade to "base"/"small" for better accuracy
print('Whisper tiny loaded')

In [ ]:
from llm.client import LLMClient
llm = LLMClient()
print(f'LLM loaded: {llm.model_name}')

In [ ]:
# Record 5 seconds via browser mic
from IPython.display import Javascript, Audio, display
from google.colab import output
import base64, os

AUDIO_PATH = '/content/user_input.wav'

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b64   = b  => new Promise(r => { const fr = new FileReader(); fr.onload = () => r(fr.result); fr.readAsDataURL(b); });
const stream = await navigator.mediaDevices.getUserMedia({audio: true});
const rec = new MediaRecorder(stream);
const chunks = [];
rec.ondataavailable = e => chunks.push(e.data);
rec.start();
await sleep(5000);
rec.stop();
await sleep(300);
const data = await b64(new Blob(chunks));
google.colab.kernel.invokeFunction('notebook.save_audio', [data], {});
"""

def save_audio(data):
    audio_bytes = base64.b64decode(data.split(',')[1])
    with open('/content/raw.webm', 'wb') as f:
        f.write(audio_bytes)
    os.system(f'ffmpeg -i /content/raw.webm -ar 16000 -ac 1 {AUDIO_PATH} -y -loglevel quiet')
    print(f'Saved: {AUDIO_PATH}')

output.register_callback('notebook.save_audio', save_audio)
display(Javascript(RECORD_JS))
print('Recording 5 seconds — speak now...')

In [ ]:
# STT + memory extraction + LLM generation
from stt.transcribe import transcribe_with_embedding
from memory.extractor import extract_turn
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve
from llm.context import build_system_prompt, BASE_COMPANION_PROMPT
import numpy as np

# Transcribe
transcript, audio_emb = transcribe_with_embedding(stt_model, AUDIO_PATH)
print(f'Transcript:  "{transcript}"')
print(f'Emb std:     {audio_emb.std():.3f}')

# Extract memories
memories = extract_turn(transcript, audio_emb, llm)
print(f'\nExtracted {len(memories)} memories:')
for m in memories:
    print(f'  [{m["emotion"]["label"]} {m["emotion"]["intensity"]:.2f}] {m["content"]}')

# Generate response
system = build_system_prompt([], BASE_COMPANION_PROMPT)
response_text = llm.generate(
    system=system,
    messages=[{'role': 'user', 'content': transcript}],
    max_tokens=100
)
print(f'\nResponse: "{response_text}"')

In [ ]:
import gc, torch
if 'stt_model' in dir(): del stt_model
if 'llm' in dir() and llm: llm.unload(); llm = None
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

from tts.model import load_tts
tts = load_tts()
print('TTS loaded (pyttsx3 — offline, no download)')

In [ ]:
from tts.synthesize import synthesize
OUTPUT_PATH = '/tmp/response.wav'
synthesize(tts, response_text, OUTPUT_PATH)
from IPython.display import Audio, display
display(Audio(OUTPUT_PATH, autoplay=True))
print('Done.')

## 7. Save Results

In [ ]:
import json, datetime
from google.colab import files

out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': 'T4',
    'llm': 'Qwen3-4B-4bit',
    'signal_noise': {k: v for k, v in hippo_result.items() if k != 'retrieved'} if 'hippo_result' in dir() else 'not run',
    'naive_rag':    {k: v for k, v in naive_result.items()  if k != 'retrieved'} if 'naive_result'  in dir() else 'not run',
    'locomo':       {'accuracy': result['accuracy']} if 'result' in dir() else 'not run',
}

with open('/content/hippovoice_results.json', 'w') as f:
    json.dump(out, f, indent=2)

files.download('/content/hippovoice_results.json')
print(json.dumps(out, indent=2))